In [1]:
# ========================================
# SECTION 1: IMPORTS AND SETUP
# ========================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
import re
import string
import os
from collections import Counter

# ML and NLP imports
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import confusion_matrix, classification_report, adjusted_rand_score, silhouette_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# NLTK imports and downloads
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

print("✅ All imports completed successfully!")

✅ All imports completed successfully!


[nltk_data] Downloading package punkt to /home/georg/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /home/georg/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/georg/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [2]:
# ========================================
# SECTION 2: DATA LOADING AND EXPLORATION
# ========================================
# Load the dataset
df = pd.read_csv('data/1429_1.csv')

/tmp/ipykernel_19814/3335042791.py:5: DtypeWarning: Columns (1,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('data/1429_1.csv')


In [3]:
# Basic dataset information
print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())
print("df.head():")

Dataset shape: (34660, 21)

Column names:
['id', 'name', 'asins', 'brand', 'categories', 'keys', 'manufacturer', 'reviews.date', 'reviews.dateAdded', 'reviews.dateSeen', 'reviews.didPurchase', 'reviews.doRecommend', 'reviews.id', 'reviews.numHelpful', 'reviews.rating', 'reviews.sourceURLs', 'reviews.text', 'reviews.title', 'reviews.userCity', 'reviews.userProvince', 'reviews.username']
df.head():


In [4]:
# Display first few rows
print("\nFirst 5 rows:")
df.head()


First 5 rows:


,id,name,asins,brand,categories,keys,manufacturer,reviews.date,reviews.dateAdded,reviews.dateSeen,...,reviews.doRecommend,reviews.id,reviews.numHelpful,reviews.rating,reviews.sourceURLs,reviews.text,reviews.title,reviews.userCity,reviews.userProvince,reviews.username
0,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-13T00:00:00.000Z,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",...,True,NaN,0.0,5.0,http://reviews.bestbuy.com/3545/5620406/review...,This product so far has not disappointed. My c...,Kindle,NaN,NaN,Adapter
1,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-13T00:00:00.000Z,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",...,True,NaN,0.0,5.0,http://reviews.bestbuy.com/3545/5620406/review...,great for beginner or experienced person. Boug...,very fast,NaN,NaN,truman
2,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-13T00:00:00.000Z,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",...,True,NaN,0.0,5.0,http://reviews.bestbuy.com/3545/5620406/review...,Inexpensive tablet for him to use and learn on...,Beginner tablet for our 9 year old son.,NaN,NaN,DaveZ
3,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-13T00:00:00.000Z,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",...,True,NaN,0.0,4.0,http://reviews.bestbuy.com/3545/5620406/review...,I've had my Fire HD 8 two weeks now and I love...,Good!!!,NaN,NaN,Shacks
4,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-12T00:00:00.000Z,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",...,True,NaN,0.0,5.0,http://reviews.bestbuy.com/3545/5620406/review...,I bought this for my grand daughter when she c...,Fantastic Tablet for kids,NaN,NaN,explore42


In [5]:
# Data structure analysis
print("DATA STRUCTURE ANALYSIS FOR CLUSTERING")
print("="*50)
print(f"Total reviews: {len(df):,}")
print(f"Unique products (by ID): {df['id'].nunique():,}")
print(f"Unique product names: {df['name'].nunique():,}")

DATA STRUCTURE ANALYSIS FOR CLUSTERING
Total reviews: 34,660
Unique products (by ID): 42
Unique product names: 48


In [6]:
# ========================================
# SECTION 3: DATA QUALITY ANALYSIS
# ========================================
# Product name and ID analysis
print(f"\nPRODUCT NAME ISSUES:")
print("Sample of product names:")
sample_names = df['name'].dropna().unique()[:10]
for i, name in enumerate(sample_names, 1):
    print(f"  {i}. {name}")

print(f"\nPRODUCT ID ANALYSIS:")
print("Sample of product IDs:")
sample_ids = df['id'].unique()[:10]
for i, product_id in enumerate(sample_ids, 1):
    print(f"  {i}. {product_id}")


PRODUCT NAME ISSUES:
Sample of product names:
  1. All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi, 16 GB - Includes Special Offers, Magenta
  2. Kindle Oasis E-reader with Leather Charging Cover - Merlot, 6 High-Resolution Display (300 ppi), Wi-Fi - Includes Special Offers,,
  3. Amazon Kindle Lighted Leather Cover,,,
Amazon Kindle Lighted Leather Cover,,,
  4. Amazon Kindle Lighted Leather Cover,,,
Kindle Keyboard,,,
  5. Kindle Keyboard,,,
Kindle Keyboard,,,
  6. All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi, 32 GB - Includes Special Offers, Magenta
  7. Fire HD 8 Tablet with Alexa, 8 HD Display, 32 GB, Tangerine - with Special Offers,
  8. Amazon 5W USB Official OEM Charger and Power Adapter for Fire Tablets and Kindle eReaders,,,
Amazon 5W USB Official OEM Charger and Power Adapter for Fire Tablets and Kindle eReaders,,,
  9. All-New Kindle E-reader - Black, 6 Glare-Free Touchscreen Display, Wi-Fi -  Includes Special Offers,,
  10. Amazon Kindle Fire Hd (3rd Generation) 8gb,,,
Amaz

In [7]:
# Category analysis
print(f"\nCATEGORY STRING ANALYSIS:")
print("Sample category strings:")
sample_categories = df['categories'].dropna().unique()[:5]
for i, cat in enumerate(sample_categories, 1):
    print(f"  {i}. {cat}")


CATEGORY STRING ANALYSIS:
Sample category strings:
  1. Electronics,iPad & Tablets,All Tablets,Fire Tablets,Tablets,Computers & Tablets
  2. eBook Readers,Kindle E-readers,Computers & Tablets,E-Readers & Accessories,E-Readers
  3. Electronics,eBook Readers & Accessories,Covers,Kindle Store,Amazon Device Accessories,Kindle E-Reader Accessories,Kindle (5th Generation) Accessories,Kindle (5th Generation) Covers
  4. Kindle Store,Amazon Devices,Electronics
  5. Tablets,Fire Tablets,Electronics,Computers,Computer Components,Hard Drives & Storage,Computers & Tablets,All Tablets


In [8]:
# Brand analysis
print(f"\nBRAND ANALYSIS:")
brand_counts = df['brand'].value_counts().head()
print("Top brands:")
for brand, count in brand_counts.items():
    print(f"  {brand}: {count:,} reviews")


BRAND ANALYSIS:
Top brands:
  Amazon: 28,701 reviews
  Amazon Fire Tv: 5,056 reviews
  Amazon Echo: 636 reviews
  Amazon Fire: 256 reviews
  Amazon Digital Services Inc.: 10 reviews


In [9]:
# ========================================
# SECTION 4: PRODUCT CATEGORIZATION
# ========================================
def categorize_product_improved(category_string, product_name):
    """
    Enhanced product categorization based on comprehensive data analysis
    """
    if pd.isna(category_string):
        category_string = ""
    if pd.isna(product_name):
        product_name = ""
    
    cat_lower = str(category_string).lower()
    prod_lower = str(product_name).lower()
    
    # Categorization logic for different product types
    ereader_terms = ['kindle', 'ebook', 'e-reader', 'kindle store', 'e-readers']
    tablet_terms = ['fire tablet', 'ipad', 'tablet', 'all tablets']
    accessory_terms = ['cover', 'accessor', 'cable', 'charger', 'case', 'adapter', 'power']
    smart_terms = ['echo', 'fire tv', 'entertainment', 'tvs entertainment', 'streaming']
    
    if any(term in cat_lower for term in ereader_terms) or any(term in prod_lower for term in ['kindle', 'ebook']):
        return "E-Readers"
    elif any(term in cat_lower for term in tablet_terms) or any(term in prod_lower for term in ['tablet', 'fire hd', 'ipad']):
        return "Tablets"
    elif any(term in cat_lower for term in accessory_terms) or any(term in prod_lower for term in ['cover', 'charger', 'cable', 'case']):
        return "Accessories"
    elif any(term in cat_lower for term in smart_terms) or any(term in prod_lower for term in ['echo', 'fire tv', 'alexa']):
        return "Smart Home & Entertainment"
    else:
        return "Other Electronics"

# Apply categorization
df['meta_category_improved'] = df.apply(
    lambda row: categorize_product_improved(row['categories'], row['name']), axis=1
)

In [10]:
# ========================================
# SECTION 5: DATA CLEANING AND PREPARATION
# ========================================
def clean_product_name(name):
    """Clean product names by removing contamination patterns like ',,,' or HTML junk."""
    if pd.isna(name):
        return None
    name = str(name)
    
    if ',,,' in name:
        name = name.split(',,,')[0]
    
    name = name.replace('\r', ' ').replace('\n', ' ').strip()
    return name

# Apply cleaning
df['clean_name'] = df['name'].apply(clean_product_name)

# Create product-level dataset for clustering
product_features = df.groupby('id').agg({
    'clean_name': 'first',
    'categories': 'first',
    'brand': 'first',
    'reviews.rating': ['mean', 'count'],
    'reviews.text': lambda x: ' '.join(x.fillna(''))
}).reset_index()

# Flatten column names
product_features.columns = [
    'product_id', 'name', 'categories', 'brand',
    'avg_rating', 'review_count', 'all_review_text'
]

print(f"Product-level dataset created: {len(product_features)} products")

Product-level dataset created: 42 products


In [11]:
# ========================================
# SECTION 6: FEATURE ENGINEERING FOR CLUSTERING
# ========================================
# Prepare numeric features
features = product_features[['avg_rating', 'review_count']].copy()
features['avg_rating'] = features['avg_rating'].fillna(features['avg_rating'].median())
features['review_count'] = features['review_count'].fillna(features['review_count'].median())
features['log_review_count'] = np.log1p(features['review_count'])

# Scale features
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features[['avg_rating', 'log_review_count']])

# Prepare text features for clustering
combined_texts = (
    product_features['categories'].fillna('') + ' ' +
    product_features['name'].fillna('')
)

vectorizer = TfidfVectorizer(
    max_features=100,
    stop_words='english',
    lowercase=True
)
X = vectorizer.fit_transform(combined_texts)

In [12]:
# ========================================
# SECTION 7: K-MEANS CLUSTERING
# ========================================
# Elbow method to find optimal K
k_range = range(2, min(11, len(product_features)))
inertias = []
sil_scores = []

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X)
    inertias.append(kmeans.inertia_)
    
    if k < len(product_features):
        sil_score = silhouette_score(X, kmeans.labels_)
        sil_scores.append(sil_score)
    else:
        sil_scores.append(0)

# Apply final clustering
optimal_k = k_range[np.argmax(sil_scores)]
final_kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
product_features['cluster'] = final_kmeans.fit_predict(X)

# Alternative clustering with K=4
alternative_kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
product_features['cluster_k4'] = alternative_kmeans.fit_predict(X)

In [13]:
# ========================================
# SECTION 8: FINAL META-CATEGORIZATION
# ========================================
def assign_final_meta_category(row):
    """Assigns final meta-category based on product name, category, and brand."""
    name = str(row['name']).lower() if pd.notna(row['name']) else ''
    categories = str(row['categories']).lower() if pd.notna(row['categories']) else ''
    brand = str(row['brand']).lower() if pd.notna(row['brand']) else ''
    
    # Categorization logic
    if (any(term in categories for term in ['ebook', 'e-reader', 'kindle e-reader']) or
        any(term in name for term in ['kindle', 'paperwhite', 'oasis', 'voyage'])) and 'fire' not in name:
        return "E-Readers"
    elif (any(term in categories for term in ['fire tablet', 'tablet']) or
          any(term in name for term in ['fire', 'tablet'])) and 'cover' not in name and 'charger' not in name:
        return "Tablets"
    elif (any(term in categories for term in ['streaming', 'fire tv', 'echo', 'speakers', 'smart']) or
          'echo' in name or 'fire tv' in brand):
        return "Smart Home & Entertainment"
    elif (any(term in categories for term in ['cover', 'accessor', 'cable', 'charger', 'adapter', 'power']) or
          any(term in name for term in ['cover', 'charger', 'cable', 'adapter'])):
        return "Accessories"
    elif 'rice dishes' in categories or 'beauty' in categories:
        return "Non-Electronics"
    else:
        return "Other Electronics"

# Apply final categorization
product_features['final_meta_category'] = product_features.apply(assign_final_meta_category, axis=1)

In [14]:
# ========================================
# SECTION 9: RESULTS EXPORT AND VISUALIZATION
# ========================================
# Create category mapping with numeric clusters
unique_categories = product_features['final_meta_category'].dropna().unique()
category_to_number = {cat: idx for idx, cat in enumerate(sorted(unique_categories))}

category_mapping = product_features[['product_id', 'name', 'brand', 'final_meta_category']].copy()
category_mapping['cluster'] = category_mapping['final_meta_category'].map(category_to_number)

# Use brand name if product name is not available
category_mapping['name'] = category_mapping.apply(
    lambda row: row['brand'] if pd.isna(row['name']) or row['name'] == '' else row['name'], 
    axis=1
)
category_mapping['name'] = category_mapping['name'].fillna('Product name not available')
category_mapping = category_mapping[['product_id', 'name', 'cluster']]

# Save results
os.makedirs('results', exist_ok=True)
output_csv = 'data/category_mapping.csv'
category_mapping.to_csv(output_csv, index=False)

print(f"\n✅ Category mapping file created! Saved to: {output_csv}")
print(f"Total products: {len(category_mapping)}")

# Display final summary
print("\nFINAL META-CATEGORY DISTRIBUTION:")
final_dist = product_features['final_meta_category'].value_counts()
for category, count in final_dist.items():
    print(f"  {category}: {count} products")

print("\n✅ CLUSTERING AND FINAL META-CATEGORIES COMPLETE!")


✅ Category mapping file created! Saved to: data/category_mapping.csv
Total products: 42

FINAL META-CATEGORY DISTRIBUTION:
  Tablets: 22 products
  E-Readers: 12 products
  Smart Home & Entertainment: 5 products
  Accessories: 2 products
  Non-Electronics: 1 products

✅ CLUSTERING AND FINAL META-CATEGORIES COMPLETE!


In [15]:
"""
Script to update cluster values in product_catalog.csv with cluster values from category_mapping.csv
"""

import pandas as pd

def update_clusters():
    """Update cluster values in product catalog using category mapping"""
    
    # Read the CSV files
    print("Reading category_mapping.csv...")
    category_mapping = pd.read_csv('data/category_mapping.csv')
    
    print("product_reviews_sentiment_and_confidence_SVM.csv...")
    product_catalog = pd.read_csv('data/product_reviews_sentiment_and_confidence_SVM.csv')
    
    print(f"Category mapping has {len(category_mapping)} rows")
    print(f"Product catalog has {len(product_catalog)} rows")
    
    # Create a dictionary mapping product_id to cluster from category_mapping
    cluster_mapping = dict(zip(category_mapping['product_id'], category_mapping['cluster']))
    
    # Update cluster values in product_catalog where product_id matches
    updated_count = 0
    for idx, row in product_catalog.iterrows():
        product_id = row['product_id']
        if product_id in cluster_mapping:
            old_cluster = row['cluster']
            new_cluster = cluster_mapping[product_id]
            if old_cluster != new_cluster:
                product_catalog.at[idx, 'cluster'] = new_cluster
                updated_count += 1
                print(f"Updated {product_id}: cluster {old_cluster} -> {new_cluster}")
    
    print(f"\nTotal updates made: {updated_count}")
    
    # Save the updated product catalog
    print("Saving updated product_catalog.csv...")
    product_catalog.to_csv('data/product_reviews_sentiment_and_confidence_SVM_updated.csv', index=False)
    
if __name__ == "__main__":
    def update_clusters():
        """Update cluster values in product catalog using category mapping"""
        
        # Read the CSV files
        print("Reading category_mapping.csv...")
        category_mapping = pd.read_csv('data/category_mapping.csv')
        
        print("Reading product_reviews_sentiment_and_confidence_SVM.csv...")
        product_catalog = pd.read_csv('data/product_reviews_sentiment_and_confidence_SVM.csv')
        
        print(f"Category mapping has {len(category_mapping)} rows")
        print(f"Product catalog has {len(product_catalog)} rows")
        print(f"Product catalog columns: {product_catalog.columns.tolist()}")
        
        # Create dictionaries mapping product_id to cluster and name from category_mapping
        cluster_mapping = dict(zip(category_mapping['product_id'], category_mapping['cluster']))
        name_mapping = dict(zip(category_mapping['product_id'], category_mapping['name']))
        
        # Add cluster and name columns if they don't exist
        if 'cluster' not in product_catalog.columns:
            print("Adding new 'cluster' column...")
            product_catalog['cluster'] = None
        
        if 'name' not in product_catalog.columns:
            print("Adding new 'name' column...")
            product_catalog['name'] = None
        
        # Update cluster and name values in product_catalog where product_id matches
        updated_count = 0
        new_assignments = 0
        name_updates = 0
        
        for idx, row in product_catalog.iterrows():
            product_id = row['product_id']
            if product_id in cluster_mapping:
                # Update cluster
                old_cluster = row['cluster']
                new_cluster = cluster_mapping[product_id]
                
                if pd.isna(old_cluster):  # New assignment
                    product_catalog.at[idx, 'cluster'] = new_cluster
                    new_assignments += 1
                    print(f"New assignment {product_id}: cluster -> {new_cluster}")
                elif old_cluster != new_cluster:  # Update existing
                    product_catalog.at[idx, 'cluster'] = new_cluster
                    updated_count += 1
                    print(f"Updated {product_id}: cluster {old_cluster} -> {new_cluster}")
                
                # Update name
                if product_id in name_mapping:
                    product_catalog.at[idx, 'name'] = name_mapping[product_id]
                    name_updates += 1
        
        print(f"\nTotal cluster updates made: {updated_count}")
        print(f"Total new cluster assignments made: {new_assignments}")
        print(f"Total product names added: {name_updates}")
        
        # Save the updated product catalog
        print("Saving updated product_catalog.csv...")
        product_catalog.to_csv('data/product_reviews_sentiment_and_confidence_SVM_updated.csv', index=False)
        
        print("Done! Check the updated file for cluster assignments.")

    update_clusters()

Reading category_mapping.csv...
Reading product_reviews_sentiment_and_confidence_SVM.csv...
Category mapping has 42 rows
Product catalog has 34627 rows
Product catalog columns: ['product_id', 'reviews.text', 'rating', 'original_sentiment_from_rating', 'predicted_sentiment_SVM', 'prediction_confidence']
Adding new 'cluster' column...
Adding new 'name' column...
New assignment AVqkIhwDv8e3D1O-lebb: cluster -> 4
New assignment AVqkIhwDv8e3D1O-lebb: cluster -> 4
New assignment AVqkIhwDv8e3D1O-lebb: cluster -> 4
New assignment AVqkIhwDv8e3D1O-lebb: cluster -> 4
New assignment AVqkIhwDv8e3D1O-lebb: cluster -> 4
New assignment AVqkIhwDv8e3D1O-lebb: cluster -> 4
New assignment AVqkIhwDv8e3D1O-lebb: cluster -> 4
New assignment AVqkIhwDv8e3D1O-lebb: cluster -> 4
New assignment AVqkIhwDv8e3D1O-lebb: cluster -> 4
New assignment AVqkIhwDv8e3D1O-lebb: cluster -> 4
New assignment AVqkIhwDv8e3D1O-lebb: cluster -> 4
New assignment AVqkIhwDv8e3D1O-lebb: cluster -> 4
New assignment AVqkIhwDv8e3D1O-lebb: 